# Agent Foundations: Interfaces, Tokenizer, and Context Window

> This notebook lays the groundwork for the agentic layer of CogitareLink, as outlined in `plans/composer-gemini-approach.md` and inspired by `plans/composer.md`.

We will define:
1.  Core `Protocol`-based interfaces for LLMs, Tools, Retrievers, Materialisers, Committers, and Tokenisers.
2.  A `Tokeniser` implementation.
3.  A `ContextWindow` manager that uses the `Tokeniser` to manage token budgets.

We will follow an exploratory approach, with `nbdev` tests for each component.

In [ ]:
#| default_exp agent.interfaces

In [ ]:
#| export
from typing import Protocol, Iterable, Mapping, Any, List, Union, Dict,TypedDict
from __future__ import annotations # Ensure forward 
from fastcore.test import *


In [ ]:
#| export
# Placeholder for actual Message type, can be a TypedDict or Pydantic model later
class Message(TypedDict):
    role: str
    content: str
    # Potentially other fields like tool_calls, tool_call_id

In [ ]:
#| export
class LLM(Protocol):
    def generate(self,
                 messages: List[Message],
                 tools: Mapping[str, "Tool"] | None = None,
                 **kwargs) -> Message: ...
    
    @property
    def tokeniser(self) -> "Tokeniser": ... # LLM should provide its tokenizer

In [ ]:
#| export
class Tool(Protocol):
    name: str
    description: str
    parameters_schema: Mapping[str, Any] # JSON Schema for parameters

    def __call__(self, **arguments) -> str: ... # Returns a string representation of the result


In [ ]:
#| export
class Retriever(Protocol):
    def fetch(self, query: str, k: int = 10, **kw) -> Iterable[Any]: ... # Returns iterable of entities/data

In [ ]:
#| export
class Materialiser(Protocol):
    def serialise(self, objects: Iterable[Any], style: str | None = None, token_budget: int | None = None) -> str: ...

In [ ]:
#| export
class Committer(Protocol):
    def ingest(self, llm_output: str, **context) -> None: ...

In [ ]:
#| export
class Tokeniser(Protocol):
    def count_tokens(self, text: str) -> int: ...
    def encode(self, text: str) -> List[int]: ...
    def decode(self, tokens: List[int]) -> str: ...


Next, we'll define a basic tokenizer. For now, we can start with a simple one or use `tiktoken` if an OpenAI dependency is acceptable early on. The `Agentic-Coder-Brief` mentions `httpx` and `pyld` but not `tiktoken` directly, so a simple word counter might be a safer first step for pure "core" functionality, with `tiktoken` as part of an optional adapter.


In [ ]:
#| export
class SimpleTokeniser(Tokeniser):
    """A basic tokenizer that splits by space and counts words."""
    def count_tokens(self, text: str) -> int:
        if not text: return 0
        return len(text.split())

    def encode(self, text: str) -> List[int]:
        # This is a placeholder; a real tokenizer would map to integer IDs.
        # For SimpleTokeniser, we're not actually creating token IDs.
        # This method is here to satisfy the protocol but won't be truly functional.
        return [len(word) for word in text.split()] # Example: return lengths of words

    def decode(self, tokens: List[int]) -> str:
        # Placeholder, as we don't have a real token-to-string mapping.
        return f"Cannot decode with SimpleTokeniser, token lengths: {tokens}"


In [ ]:
# Tests
_st = SimpleTokeniser()
test_eq(_st.count_tokens("Hello world"), 2)
test_eq(_st.count_tokens(""), 0)
test_eq(_st.count_tokens("One"), 1)

Now, let's implement the `ContextWindow`. It will use a `Tokeniser` to manage its contents based on a token budget.


In [ ]:
#| export

# Assuming interfaces are available from cogitarelink.agent.interfaces
# from cogitarelink.agent.interfaces import Tokeniser, Message # Would be imported in the .py file
# Assuming SimpleTokeniser is available from cogitarelink.core.tokenizer
# from cogitarelink.core.tokenizer import SimpleTokeniser # Would be imported in the .py file


class ContextWindow:
    def __init__(self, max_tokens: int, tokeniser: Tokeniser, initial_fragments: List[Message] | None = None):
        self.max_tokens = max_tokens
        self.tokeniser = tokeniser
        self.fragments: List[Message] = []
        self.tokens_used = 0

        if initial_fragments:
            for fragment in initial_fragments:
                self.add(fragment) # Use self.add to correctly count tokens

    def add(self, fragment: Message):
        # For simplicity, we'll assume fragment['content'] is the main text for token counting.
        # A more sophisticated Message type might have specific fields for tokenization.
        text_to_add = fragment['content']
        tokens_for_fragment = self.tokeniser.count_tokens(text_to_add)
        
        if tokens_for_fragment > self.max_tokens:
            # Fragment itself is too large, decide on handling (e.g., truncate, error)
            # For now, let's skip adding it and perhaps log a warning.
            print(f"Warning: Fragment too large ({tokens_for_fragment} tokens) for max_tokens ({self.max_tokens}). Skipping.")
            return

        if self.tokens_used + tokens_for_fragment > self.max_tokens:
            self._evict(self.tokens_used + tokens_for_fragment - self.max_tokens)
        
        self.fragments.append(fragment)
        self.tokens_used += tokens_for_fragment

    def _evict(self, tokens_to_free: int):
        """Simple FIFO eviction; swap-in richer policy later."""
        freed_tokens = 0
        while self.fragments and freed_tokens < tokens_to_free:
            removed_fragment = self.fragments.pop(0)
            # Assume content is the key for token counting
            freed_tokens += self.tokeniser.count_tokens(removed_fragment['content'])
        self.tokens_used -= freed_tokens
        if self.tokens_used < 0: self.tokens_used = 0 # Ensure it doesn't go negative

    def render_messages(self) -> List[Message]:
        return list(self.fragments) # Return a copy

    def __repr__(self):
        return f"<ContextWindow tokens_used={self.tokens_used}/{self.max_tokens}, num_fragments={len(self.fragments)}>"


In [ ]:
_tokeniser = SimpleTokeniser()
_cw = ContextWindow(max_tokens=10, tokeniser=_tokeniser)
_cw.add({"role": "system", "content": "You are a helpful assistant."}) # 5 tokens
test_eq(_cw.tokens_used, 5)
test_eq(len(_cw.fragments), 1)


In [ ]:
_cw.add({"role": "user", "content": "Tell me a joke."}) # 4 tokens
test_eq(_cw.tokens_used, 9)
test_eq(len(_cw.fragments), 2)


In [ ]:
 # This should cause eviction of the first message
_cw.add({"role": "assistant", "content": "Why did the chicken cross the road?"}) # 7 tokens. 9+7=16. Needs to free 16-10=6. First msg (5 tokens) is not enough.
# Ah, my eviction logic needs to be more precise or the test needs adjustment.
# If "You are a helpful assistant." (5 tokens) is evicted, tokens_used becomes 9-5=4.
# Then add "Why did the chicken cross the road?" (7 tokens). 4+7=11. Still > 10.
# This means the second message "Tell me a joke." (4 tokens) must also be evicted.
# Let's re-evaluate the eviction logic or test case.


In [ ]:
# Revised eviction test logic:
_tokeniser_test = SimpleTokeniser()
_cw_test = ContextWindow(max_tokens=10, tokeniser=_tokeniser_test)
_cw_test.add({"role": "system", "content": "Fragment one two three four five"}) # 5 tokens
_cw_test.add({"role": "user", "content": "Fragment six seven eight nine"})    # 4 tokens, total 9
test_eq(_cw_test.tokens_used, 9)
_cw_test.add({"role": "assistant", "content": "Fragment ten eleven twelve"}) # 3 tokens. 9+3=12. Needs to free 2.
# Evicts "Fragment one two three four five" (5 tokens). tokens_used = 9-5=4.
# Now adds "Fragment ten eleven twelve" (3 tokens). tokens_used = 4+3=7.
test_eq(_cw_test.tokens_used, 7)
test_eq(len(_cw_test.fragments), 2)
test_eq(_cw_test.fragments[0]['content'], "Fragment six seven eight nine")


AssertionError: ==:
5
9